# 🔑 Banco de Dados Chave-Valor (NoSQL em Memória)

> **Missão:** Comprovar a velocidade absoluta do acesso matemático $O(1)$ de uma Tabela Hash contra o desastre de tentar varrer a memória procurando por um valor desconhecido $O(N)$.
> **Arquivo:** Simularemos um dump de memória gravado no arquivo `cache_sessoes.json`.

Em sistemas reais, usamos esse banco para guardar sessões de login e carrinhos de compras temporários. O acesso pela **Chave** é instantâneo. Mas se você perder a chave e tentar buscar pelo **Valor**, o banco precisará escanear tudo.

In [1]:
import json
import os
import random

file_path = 'cache_sessoes.json'

if not os.path.exists(file_path):
    print("⏳ Gerando 'cache_sessoes.json' no disco (isso pode levar alguns segundos)...")
    
    # Gerando 1.000.000 de chaves-valor simples
    dados = {}
    for i in range(1, 1000001):
        # Chave: "sessao:user_X" | Valor: "token_aleatorio"
        dados[f"sessao:user_{i}"] = f"token_{random.randint(100000, 999999)}_{i}"
        
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(dados, f)
        
    print("✅ Arquivo 'cache_sessoes.json' criado e salvo com sucesso na pasta!")
else:
    print("✅ Arquivo 'cache_sessoes.json' já existe. Pronto para as consultas!")

⏳ Gerando 'cache_sessoes.json' no disco (isso pode levar alguns segundos)...
✅ Arquivo 'cache_sessoes.json' criado e salvo com sucesso na pasta!


In [3]:
import time
import json
import ipywidgets as widgets
from ipywidgets import interact

# 1. Carregamento dos dados para a "RAM" (Dicionário Python)
with open('cache_sessoes.json', 'r', encoding='utf-8') as f:
    redis_cache = json.load(f)

# 2. Função de Visualização
def exibir_resultado(query: str, titulo: str, explicacao: str, tempo_ms: float, chaves_lidas: int, resultado: str):
    print(f"\n{'=' * 80}")
    print("⚙️  PAINEL DE TESTES NOSQL (CHAVE-VALOR)")
    print(f"{'=' * 80}")

    print("\n💻 COMANDO EXECUTADO")
    print(f"{'-' * 80}")
    print(query.strip())

    print("\n📊 RESUMO")
    print(f"{'-' * 80}")
    print(f"Tempo de execução : {tempo_ms:.5f} ms") # 5 casas decimais pois é ultra rápido
    print(f"Chaves lidas      : {chaves_lidas:,}".replace(",", "."))
    print(f"Resultado         : {titulo}")

    print("\n📝 O QUE ACONTECEU?")
    print(f"{'-' * 80}")
    print(explicacao)

    print("\n📋 RESULTADO DA CONSULTA")
    print(f"{'-' * 80}")
    
    if not resultado:
        print("Nenhum dado encontrado.")
    else:
        print(f"Valor retornado: {resultado}")

    print(f"\n{'=' * 80}")


# 3. Lógica de Execução
def testar_chave_valor(tipo_query: str):
    inicio = time.time()

    if tipo_query == "Eficiente (Busca direta pela Chave)":
        query = """
                GET sessao:user_850000
                """
        # Acesso O(1) via Hash Table
        resultado = redis_cache.get("sessao:user_850000")
        
        titulo = "🚀 Acesso Matemático O(1)"
        explicacao = (
            "O banco converteu a chave em um endereço físico de memória usando "
            "uma fórmula matemática (Hash). Ele não procurou o dado, ele simplesmente "
            "foi até a 'gaveta' correta e pegou. O tempo beira zero milissegundos."
        )
        chaves_lidas = 1
        
    else:
        token_alvo = redis_cache.get("sessao:user_850000") # Pegamos um token real para buscar
        query = f"""
                SCAN iterando para encontrar VALOR == '{token_alvo}'
                """
        # Acesso O(N) varrendo os valores
        resultado = None
        for chave, valor in redis_cache.items():
            if valor == token_alvo:
                resultado = chave
                break # Para a busca quando encontrar
                
        titulo = "🚨 Varredura Linear O(N)"
        explicacao = (
            "Você pediu para o banco procurar por um 'Valor'. Como os valores "
            "não são indexados pela fórmula Hash, o banco precisou escanear a "
            "memória verificando chave por chave até encontrar uma correspondência."
        )
        chaves_lidas = 850000 # Leu até encontrar no índice 850 mil

    fim = time.time()
    tempo_ms = (fim - inicio) * 1000

    exibir_resultado(query, titulo, explicacao, tempo_ms, chaves_lidas, resultado)


# 4. Interface Gráfica
opcoes_cenarios = [
    "Eficiente (Busca direta pela Chave)",
    "Ineficiente (Busca escaneando pelo Valor)"
]

interact(
    testar_chave_valor,
    tipo_query=widgets.RadioButtons(
        options=opcoes_cenarios,
        description="Cenário:",
        layout={'width': 'max-content'}
    )
)

interactive(children=(RadioButtons(description='Cenário:', layout=Layout(width='max-content'), options=('Efici…

<function __main__.testar_chave_valor(tipo_query: str)>